In [8]:
import requests
import json
import pandas as pd
import numpy as np
import os

API_KEY = None
with open("keys.json") as f:
    data = json.loads(f.read())
    API_KEY = data["STEAM_WEB_API"] 

BASE_URL = "https://www.steamwebapi.com/"
MARKET_IDENTS = "steam,49skins,buff,csdeals,csfloat,csgocom,dmarket,haloskins,lisskins,skinland,skinbaron,skinflow,skinport,tradeit,waxpeer,youpin"

In [ ]:
def call(method: str, call_path: str, extra_params: str | None = None, payload: dict | None = None, base_url = BASE_URL):
    """
    calls the API, stores the response to /data, returns (location of saved file, errorcode)
    """
    method = method.upper()
    body = ""
    headers = {}

    request_url = base_url + call_path + "?key=" + API_KEY 

    output_path = f"data/{call_path.replace('/', '_')}"
    output_path = output_path[:output_path.find("markets")].rstrip()

    if extra_params != None:
        request_url = request_url + "&" + extra_params
        output_path = output_path + "_" + extra_params + ".json"

    if payload:
        body = json.dumps(payload)


    if method != "GET" and payload:
        headers["Content-Type"] = "application/json"

    try:
        r = requests.request(
            method,
            request_url,
            headers=headers,
            data=body.encode() if body else None,
            timeout=15,
        )
        r.raise_for_status()
        data_json = r.json()

        if len(data_json) == 0:
            return output_path, "RESPONSE EMPTY"

        file_opening_mode = "x"

        if (os.path.exists(output_path)):
            file_opening_mode = "w"
        
        with open(output_path, file_opening_mode) as f:
            json.dump(data_json, f, indent=4)

        return output_path, None
    except requests.RequestException as e:
        detail = e.response.text if e.response is not None else "no body"
        return output_path, f"{e} — {detail}"


def getJsonFileasFlatMappedDataFrame(file_path: str):
    return pd.json_normalize(json.loads(open(file_path).read()))

In [ ]:
call_path = "steam/api/items"

for game_name in ["cs2", "dota", "tf2"]:
    game_param = f"game={game_name}"
    for pagenum in range(0,5):
        full_params = game_param + '&pretty=1&' + 'page=' + str(pagenum)
        location, errorcode = call("GET", call_path, full_params)
        if (errorcode != None):
            print(errorcode)
            break
        print(f"Nr: {pagenum}, File: {location}" +  ", Size: " + str(os.path.getsize(location)/1000) + "KB" )

In [ ]:
call_path = "steam/api/items/history"
call_payload = {
  "items": [
    "AK-47 | Redline (Field-Tested)",
  ],
  "markets": MARKET_IDENTS,
  "game": "cs2",
#  "from_date": "2024-01-01",
#  "to_date": "2026-11-13",  -> getting full price history
  "strategy": "STRICT"
}

location, errorcode = call("POST", call_path, payload=call_payload)
print(errorcode)
print(f"File: {location}" +  ", Size: " + str(os.path.getsize(location)/1000) + "KB" )

None
File: data/items_history, Size: 462.936MB


In [ ]:
call_path = "steam/api/markets/history"

item_hash_name= "StatTrak\u2122 AK-47 | Fire Serpent (Field-Tested)"

full_params = f"market_hash_name={item_hash_name}"

diff_base_url = "https://www.steamwebapi.com/"
location, errorcode = call("GET", call_path=call_path, extra_params=full_params, base_url=diff_base_url)
print(errorcode)

# TODO: This call for all Items, yay

None


In [6]:
jsondf2 = json.loads(open("data/items_game=cs2&page=0.json").read())

cs2df = pd.DataFrame(jsondf2)

In [ ]:
call_path = "steam/api/items"
params = "game=cs2&pretty=1&production=1&limit=100&search=AK-47&markets=buff,steam"
location, errorcode = call("GET", call_path, params)
print(location)
print(errorcode)

data/items_game=cs2&pretty=1&production=1&limit=100&search=AK-47&markets=buff,steam.json
None


In [ ]:


full_df = getJsonFileasFlatMappedDataFrame("data/items_game=cs2&pretty=1&production=0&limit=100.json")
production_df = getJsonFileasFlatMappedDataFrame("data/items_game=cs2&pretty=1&production=1&limit=100.json")


In [10]:
call_path = "markets/prices"
params = "markets="+MARKET_IDENTS
location, errorcode = call("GET", call_path, params)
print(location)
print(errorcode)

data/markets_price_markets=steam,49skins,buff,csdeals,csfloat,csgocom,dmarket,haloskins,lisskins,skinland,skinbaron,skinflow,skinport,tradeit,waxpeer,youpin.json
None
